# Phase 2 - B3 DER++ (Kaggle)

Runs all five fixed XNLI orders with the shared Phase 1 candidate-label prompt and LoRA trainer. This notebook implements a documented causal-LM adaptation of DER++: examples entering a 125-example uniform reservoir store the pre-update model's raw next-token logits for the first token of each of the three candidate labels. Later stages sample one replay example with replacement per current example. The training objective is current answer-token CE + 0.5 * replay answer-token CE + 0.5 * MSE between stored and current three-candidate logits. This is not full-vocabulary DER++; the exact adaptation is recorded with each result.

## Before running
1. Enable a Kaggle GPU and Internet.
2. Accept the gated model terms for `meta-llama/Llama-3.2-1B` and add a Kaggle secret named `HF_TOKEN`.
3. The GitHub repository is public; no GitHub token is required to clone it.
4. Add optional `WANDB_API_KEY` only if you enable W&B.
5. The B3 runner must be committed and pushed to the public repository before Kaggle can clone it.
6. Run all cells, then use **Save Version** to persist `/kaggle/working/phase2_artifacts`. A failed attempt's checkpoints and ledgers are preserved; completed order results are validated and skipped on rerun.

This notebook runs all five B3 orders and writes `B3_summary.json` after validating every completed result. Replay increases training exposure relative to B0; compare and report that difference. Never tune using test scores.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/Mani-S-49/asymmetric-interference-continual-learning..git"
REPO_ROOT = Path("/kaggle/working/asymmetric-interference-continual-learning")

if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)

subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_ROOT)], check=True)
repo_revision = subprocess.check_output(["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], text=True).strip()
print("Cloned repository revision:", repo_revision)

requirements_file = REPO_ROOT / "phase2" / "requirements-phase2.txt"
if not requirements_file.exists():
    raise FileNotFoundError(f"Requirements file not found: {requirements_file}")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_file)], check=True)
sys.path.insert(0, str(REPO_ROOT))

runner_path = REPO_ROOT / "phase2" / "run_baseline.py"
if not runner_path.is_file():
    raise FileNotFoundError(f"Missing B3 runner: {runner_path}")

help_result = subprocess.run(
    [sys.executable, "-m", "phase2.run_baseline", "--help"],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
)
print(help_result.stdout)
print(help_result.stderr)
if help_result.returncode != 0:
    raise RuntimeError("phase2.run_baseline failed to start. Check the repo/installation.")
if "B3" not in help_result.stdout:
    raise RuntimeError(
        "The GitHub revision cloned above does not include B3 support. "
        "Commit and push the B3 runner changes to the repository's default branch, "
        "then restart this Kaggle session and rerun the notebook. The local project "
        "copy alone is not available to Kaggle."
    )

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
if not hf_token:
    raise RuntimeError("HF_TOKEN Kaggle secret is empty.")
os.environ["HF_TOKEN"] = hf_token
del hf_token

USE_WANDB = False
WANDB_PROJECT = "phase2-xnli-baselines"
if USE_WANDB:
    wandb_key = secrets.get_secret("WANDB_API_KEY")
    if not wandb_key:
        raise RuntimeError("Enable USE_WANDB only after adding the WANDB_API_KEY Kaggle secret.")
    os.environ["WANDB_API_KEY"] = wandb_key
    del wandb_key

PROTOCOL_PATH = REPO_ROOT / "phase2" / "configs" / "pilot_protocol.json"
ARTIFACT_ROOT = Path("/kaggle/working/phase2_artifacts")
print({"repo": str(REPO_ROOT), "artifact_root": str(ARTIFACT_ROOT), "runner_file": str(runner_path), "wandb": USE_WANDB})


In [ ]:
from phase2.preflight import collect_preflight
from phase2.prepare_data import prepare_xnli
from phase2.protocol import build_manifest, load_protocol
from phase2.run_baseline import validate_prepared_artifacts
import json

preflight = collect_preflight(require_cuda=True, check_gated_model=True)
print(json.dumps(preflight, indent=2, sort_keys=True))

protocol = load_protocol(PROTOCOL_PATH)
if not (ARTIFACT_ROOT / "protocol_manifest.json").is_file():
    if (ARTIFACT_ROOT / "data").exists():
        raise FileExistsError("A partial data directory exists without its manifest. Inspect it before using a fresh artifact root.")
    prepared_manifest = prepare_xnli(PROTOCOL_PATH, ARTIFACT_ROOT)
    print("Prepared fixed XNLI subsets:", prepared_manifest["expected_run_count"], "scheduled runs")
else:
    prepared_manifest = validate_prepared_artifacts(protocol, ARTIFACT_ROOT)
    print("Verified existing fixed data manifest:", prepared_manifest["protocol_sha256"])

ORDER_IDS = [order["id"] for order in protocol["orders"]]
if len(ORDER_IDS) != 5:
    raise ValueError(f"B3 requires all five protocol orders; found {len(ORDER_IDS)}.")
print({"languages": protocol["dataset"]["languages"], "orders_to_run": ORDER_IDS, "buffer_capacity": protocol["replay"]["buffer_capacity_examples"], "data_manifest_sha256": prepared_manifest["data_manifest_sha256"]})


In [ ]:
import subprocess
import sys

command = [
    sys.executable,
    "-m",
    "phase2.run_baseline",
    "--method", "B3",
    "--protocol", str(PROTOCOL_PATH),
    "--artifact-root", str(ARTIFACT_ROOT),
]
for order_id in ORDER_IDS:
    command.extend(["--order-id", order_id])
if USE_WANDB:
    command.extend(["--wandb-project", WANDB_PROJECT])
print("RUN COMMAND:")
print(command)
subprocess.run(command, cwd=REPO_ROOT, check=True)


In [ ]:
from phase2.protocol import build_manifest
from phase2.results import validate_result_record
import statistics
import json

run_manifest = build_manifest(protocol)
run_manifest["data_artifacts"] = prepared_manifest["data_artifacts"]
run_manifest["data_manifest_sha256"] = prepared_manifest["data_manifest_sha256"]
expected_runs = {run["run_id"]: run for run in run_manifest["runs"] if run["method_id"] == "B3"}
if len(expected_runs) != 5:
    raise ValueError(f"Expected five B3 runs in the current manifest; found {len(expected_runs)}.")
completed = {}
for run_id, run in expected_runs.items():
    result_path = ARTIFACT_ROOT / "results" / f"{run_id}.json"
    if not result_path.is_file():
        raise FileNotFoundError(f"B3 order is incomplete: {result_path}")
    record = json.loads(result_path.read_text(encoding="utf-8"))
    validate_result_record(record, run_manifest)
    completed[run["order_id"]] = record

metric_names = ("ACC", "BWT", "FWT", "F")
summary = {
    "method_id": "B3",
    "protocol_id": run_manifest["protocol_id"],
    "protocol_sha256": run_manifest["protocol_sha256"],
    "data_manifest_sha256": run_manifest["data_manifest_sha256"],
    "completed_orders": len(completed),
    "expected_orders": len(expected_runs),
    "per_order": {order_id: {name: record["metrics"][name] for name in metric_names} for order_id, record in sorted(completed.items())},
    "across_order_summary": {
        name: {
            "mean": statistics.mean(record["metrics"][name] for record in completed.values()),
            "population_std": statistics.pstdev(record["metrics"][name] for record in completed.values()),
        }
        for name in metric_names
    },
}
summary_path = ARTIFACT_ROOT / "B3_summary.json"
temporary_path = summary_path.with_suffix(".json.tmp")
temporary_path.write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8")
temporary_path.replace(summary_path)
print(json.dumps(summary, indent=2, sort_keys=True))
print("Validated all five B3 orders. Summary:", summary_path)


## Saved outputs

- `phase2_artifacts/protocol_manifest.json` and `data/`: frozen selected examples and checksums.
- `phase2_artifacts/checkpoints/<run_id>/`: adapter/tokenizer after each language stage.
- `phase2_artifacts/ledgers/<run_id>/`: complete stage-by-language evaluation matrix and CSV ledger.
- `phase2_artifacts/results/<run_id>.json`: validated Phase 2 result record for aggregation.
- `phase2_artifacts/B3_summary.json`: summary of all five validated orders and across-order metrics.

B3 is a causal-LM adaptation: it distills only the raw next-token logits of the three candidate-label first tokens, rather than full-vocabulary logits. The fixed loss is current CE + 0.5 x replay CE + 0.5 x three-logit MSE. Its `BWT`, `FWT`, and forgetting metrics use the complete fixed-test performance matrix. Do not select or tune settings using test scores.
